# Limpieza de datos — TalentHub

**Cliente:** TalentHub quiere lanzar un bootcamp tech y necesita saber qué habilidades piden hoy las empresas.

**Objetivo** tomar el dataset crudo de ofertas de trabajo (112.816 ofertas de todas las áreas), quedarnos con los 5 roles del caso, limpiarlas y guardar el resultado.

**Orden de trabajo** filtrar roles → estado inicial → duplicados → normalizar → nulos → tipos → estado final → guardar.


## 0. Carga de datos
Fuente: Kaggle, "Global Job Postings Multi-ATS (June 2026)"

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 50)

RUTA_RAW = "../data/raw/nextgig_jobs_2026-06.parquet"
df_raw = pd.read_parquet(RUTA_RAW)

print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")

Filas: 112,816 | Columnas: 47


## 1. Filtrar los 5 roles del caso

In [2]:
roles = {
    "Data Analyst":         ["data analyst", "analytics analyst", "bi analyst", "business intelligence analyst"],
    "Data Scientist":       ["data scientist"],
    "Full Stack Developer": ["full stack", "fullstack"],
    "QA Engineer":          [" qa ", "quality assurance", "test engineer", " sdet ", "test automation"],
    "Cloud Engineer":       ["cloud engineer", "cloud architect", "devops", "site reliability", " sre ", "platform engineer"],
}

def limpiar_titulo(titulo):
    """Pasa el título a minúsculas y cambia los símbolos por espacios."""
    texto = titulo.lower()
    for simbolo in ["-", "/", ",", "(", ")", "|", ":"]:
        texto = texto.replace(simbolo, " ")
    return " " + texto + " "


def asignar_rol(titulo):
    """Devuelve el rol de la oferta según su título, o 'Otro' si no es de los 5."""
    texto = limpiar_titulo(titulo)
    for rol, palabras_clave in roles.items():
        for clave in palabras_clave:
            if clave in texto:
                return rol
    return "Otro"


# Probamos la función antes de usarla con todo el dataset
print(asignar_rol("Senior Data Analyst"))
print(asignar_rol("Full-Stack Developer (React/Node)"))
print(asignar_rol("QA Engineer"))
print(asignar_rol("Cashier"))



Data Analyst
Full Stack Developer
QA Engineer
Otro


In [3]:
df_raw["rol"] = [asignar_rol(t) for t in df_raw["title"]]

df_raw["rol"].value_counts()

rol
Otro                    111416
QA Engineer                410
Cloud Engineer             355
Full Stack Developer       280
Data Analyst               187
Data Scientist             168
Name: count, dtype: int64

### 1.1 Validación: ¿todos los "QA" son de software?

quedarnos con las ofertas que sirven

In [4]:
terminos_software = ["software", "automation", "automated", "sdet", "selenium", "playwright", "cypress", "appium",
                     "api", "python", "java", "javascript", "typescript", "sql", "jira", "agile", "scrum", "postman",
                     "regression", "web", "mobile", "ci/cd", "jenkins", "git", "manual testing", "test cases", "test plan"]

terminos_industria = ["hardware", "silicon", "semiconductor", "electromagnetic", "manufacturing", "facilities",
                      "vehicle", "calibration", "clinical", "inspector", "technician", "pumps", "on the floor", "food", "pharma"]


def contiene_alguno(texto, terminos):
    """Devuelve True si el texto contiene al menos uno de los términos."""
    texto = texto.lower()
    for termino in terminos:
        if termino in texto:
            return True
    return False


# Juntamos título + skills en un solo texto (si no hay skills, usamos texto vacío)
df_raw["texto_qa"] = df_raw["title"] + " " + df_raw["skills_required"].fillna("")

df_raw["menciona_software"] = [contiene_alguno(t, terminos_software) for t in df_raw["texto_qa"]]
df_raw["titulo_industrial"] = [contiene_alguno(t, terminos_industria) for t in df_raw["title"]]




In [5]:
es_qa = df_raw["rol"] == "QA Engineer"
qa_descartado = es_qa & (~df_raw["menciona_software"] | df_raw["titulo_industrial"])

print(f"QA de software: {(es_qa & ~qa_descartado).sum()} | QA descartados: {qa_descartado.sum()}")
print("\nTítulos descartados más frecuentes:")
print(df_raw[qa_descartado]["title"].value_counts().head(15))

QA de software: 187 | QA descartados: 223

Títulos descartados más frecuentes:
title
Senior Hardware Test Engineer                          4
Quality Assurance Specialist                           4
Quality Assurance Technician                           4
Quality Assurance Manager                              3
QA Inspector (Pumps)                                   3
Quality Assurance Engineer                             3
Quality Assurance Coordinator                          3
Sr Quality Assurance Engineer                          3
Quality Assurance Supervisor                           2
Manager, Quality Assurance                             2
Quality Assurance Inspector                            2
Senior Test Engineer                                   2
Technician 1, Manufacturing Test Engineering           2
Technician 1, Manufacturing Test Engineer              2
Quality Assurance Specialist - After School Program    2
Name: count, dtype: int64


## 2. Elegir las columnas útiles

In [6]:
df = df_raw[(df_raw["rol"] != "Otro") & ~qa_descartado].copy()

print(f"Ofertas para analizar: {len(df):,} (de {len(df_raw):,} originales)")
df["rol"].value_counts()

Ofertas para analizar: 1,177 (de 112,816 originales)


rol
Cloud Engineer          355
Full Stack Developer    280
Data Analyst            187
QA Engineer             187
Data Scientist          168
Name: count, dtype: int64

In [7]:
columnas = ["title", "rol", "company_name", "country", "city", "work_model", "experience_level",
            "job_level_normalized", "employment_type", "skills_required", "education_level",
            "salary_min", "salary_max", "salary_currency", "salary_rate_unit", "date_posted"]

df = df[columnas].copy()
print(df.shape)

(1177, 16)


## 3. Estado inicial

In [8]:
print("=== ESTADO INICIAL ===")
print(f"Filas: {len(df)}")

porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(1)
print(f"\nNulos por columna (%):\n{porcentaje_nulos}")

print(f"\nDuplicados exactos: {df.duplicated().sum()}")

=== ESTADO INICIAL ===
Filas: 1177

Nulos por columna (%):
title                    0.0
rol                      0.0
company_name             0.0
country                 20.6
city                     9.5
work_model              25.3
experience_level        20.6
job_level_normalized    27.6
employment_type          1.1
skills_required         10.3
education_level         89.8
salary_min              61.0
salary_max              61.7
salary_currency         54.8
salary_rate_unit        60.7
date_posted             78.0
dtype: float64

Duplicados exactos: 3


## 4. Duplicados

In [9]:
claves = ["title", "company_name", "city"]

print(f"Duplicados por título + empresa + ciudad: {df.duplicated(subset=claves).sum()}")

repetidas = df[df.duplicated(subset=claves, keep=False)]
print("\nEmpresas con más ofertas repetidas:")
print(repetidas["company_name"].value_counts().head(5))

Duplicados por título + empresa + ciudad: 102

Empresas con más ofertas repetidas:
company_name
Bright Vision Technologies                    63
Accenture                                     13
BPS OPS                                       10
PricewaterhouseCoopers                        10
KPMG Assurance and Consulting Services LLP     7
Name: count, dtype: int64


In [10]:
filas_antes = len(df)
df = df.drop_duplicates(subset=claves)

print(f"Filas: {filas_antes} → {len(df)} (se eliminaron {filas_antes - len(df)})")
df["rol"].value_counts()

Filas: 1177 → 1075 (se eliminaron 102)


rol
Cloud Engineer          310
Full Stack Developer    255
Data Analyst            181
QA Engineer             173
Data Scientist          156
Name: count, dtype: int64

### 5. Normalizar categorías

 ### 5.1 Países 

In [11]:
print(f"Países distintos: {df['country'].nunique()}")
df["country"].value_counts()

Países distintos: 83


country
India                       184
USA                         176
United States               116
US                           63
United States of America     36
                           ... 
BG                            1
Ecuador                       1
PL                            1
SG                            1
Italy                         1
Name: count, Length: 83, dtype: int64

In [12]:
mapa_paises = {
    # Estados Unidos, escrito de muchas formas
    "USA": "United States", "US": "United States", "U.S": "United States",
    "United States of America": "United States",
    # Códigos de estados de EE.UU. cargados como si fueran países
    "OH": "United States", "KY": "United States", "IA": "United States",
    "MN": "United States", "VA": "United States",
    # Códigos de países
    "IN": "India", "IND": "India", "UK": "United Kingdom",
    "FR": "France", "NL": "Netherlands", "RO": "Romania", "LT": "Lithuania",
    "IE": "Ireland", "ES": "Spain", "PL": "Poland", "BG": "Bulgaria", "LUX": "Luxembourg",
    "MX": "Mexico", "MEX": "Mexico", "México": "Mexico",
    "ARG": "Argentina", "PE": "Peru", "CAN": "Canada",
    "MY": "Malaysia", "PH": "Philippines", "KR": "South Korea", "HK": "Hong Kong",
    "SG": "Singapore", "Viet Nam": "Vietnam", "UAE": "United Arab Emirates",
     "IL": "United States",
    "Europe": "Sin datos",
}

df["country"] = df["country"].replace(mapa_paises)

print(f"Países distintos: {df['country'].nunique()}")
df["country"].value_counts().head(50)

Países distintos: 50


country
United States           398
India                   189
United Kingdom           49
Canada                   34
France                   32
Germany                  18
Spain                    14
Mexico                   14
Romania                  12
Ireland                  10
Netherlands              10
Vietnam                   8
Australia                 7
Poland                    6
Singapore                 6
Argentina                 5
Bulgaria                  4
Portugal                  4
Lithuania                 4
Philippines               4
South Korea               4
Malaysia                  3
Peru                      3
Brazil                    3
Serbia                    3
Hungary                   3
South Africa              3
Chile                     3
Israel                    3
United Arab Emirates      2
Uruguay                   2
Thailand                  2
Slovakia                  2
Switzerland               2
Taiwan                    2
Costa Rica  

### 5.2 Modalidad

In [13]:
df["work_model"].value_counts(dropna=False)

work_model
On-site    396
NaN        262
Hybrid     248
Remote     169
Name: count, dtype: int64

5.3 Seniority

In [14]:
print(f"Seniority sin dato (antes): {df['experience_level'].isnull().sum()}")

# Si experience_level está vacío, usamos job_level_normalized, que mide lo mismo
df["experience_level"] = df["experience_level"].fillna(df["job_level_normalized"])

print(f"Seniority sin dato (después): {df['experience_level'].isnull().sum()}")

Seniority sin dato (antes): 227
Seniority sin dato (después): 205


In [15]:
mapa_seniority = {
    "Intern":    "Pasante",
    "Entry":     "Junior",
    "Junior":    "Junior",
    "Mid":       "Semi Senior",
    "Senior":    "Senior",
    "Principal": "Senior",
    "Staff":     "Senior",
    "10+":       "Senior",
    "Lead":      "Lead / Gerencia",
    "Director":  "Lead / Gerencia",
}

df["seniority"] = df["experience_level"].map(mapa_seniority)

# Ya no necesitamos las dos columnas originales
df = df.drop(columns=["experience_level", "job_level_normalized"])

df["seniority"].value_counts(dropna=False)

seniority
Semi Senior        557
Senior             269
NaN                205
Junior              22
Pasante             18
Lead / Gerencia      4
Name: count, dtype: int64

5.4 Tipo de contrato

In [16]:
mapa_contrato = {"FULL_TIME": "Full-time", "Permanent": "Full-time", "Contractor": "Contract"}
df["employment_type"] = df["employment_type"].replace(mapa_contrato)

df["employment_type"].value_counts(dropna=False)

employment_type
Full-time     1025
Contract        20
Internship      14
NaN             13
Part-time        3
Name: count, dtype: int64

## 6. Tratamiento de nulos

In [17]:
porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(1)
porcentaje_nulos.sort_values(ascending=False)

education_level     90.0
date_posted         77.8
salary_max          62.5
salary_min          62.0
salary_rate_unit    61.5
salary_currency     55.7
work_model          24.4
seniority           19.1
country             17.6
skills_required     10.6
city                 8.8
employment_type      1.2
rol                  0.0
company_name         0.0
title                0.0
dtype: float64

In [18]:
df["sin_modalidad"] = df["work_model"].isnull()
df["sin_seniority"] = df["seniority"].isnull()

print("% de ofertas sin dato, por rol:")
print((df.groupby("rol")[["sin_modalidad", "sin_seniority"]].mean() * 100).round(1))

df = df.drop(columns=["sin_modalidad", "sin_seniority"])

% de ofertas sin dato, por rol:

                      sin_modalidad  sin_seniority
rol                                               
Cloud Engineer                 23.5           17.4
Data Analyst                   22.7           21.5
Data Scientist                 28.8           12.2
Full Stack Developer           25.9           21.6
QA Engineer                    21.4           22.0


 rellenar con “Sin datos”  
 fillna("Sin datos")

In [19]:
for columna in ["country", "city", "work_model", "seniority"]:
    df[columna] = df[columna].fillna("Sin datos")

rellenar con la moda 
rellenar con el valor más común · clase 11: mode()

In [20]:
moda_contrato = df["employment_type"].mode()[0]
print(f"Moda de employment_type: {moda_contrato}")

df["employment_type"] = df["employment_type"].fillna(moda_contrato)

Moda de employment_type: Full-time

marcar las ofertas sin skills 
clase 09: isnull() -  clase 12: ~

In [21]:
df["tiene_skills"] = ~df["skills_required"].isnull()

print(df["tiene_skills"].value_counts())

tiene_skills
True     961
False    114
Name: count, dtype: int64


verificar

In [22]:
df.isnull().sum()

title                 0
rol                   0
company_name          0
country               0
city                  0
work_model            0
employment_type       0
skills_required     114
education_level     968
salary_min          666
salary_max          672
salary_currency     599
salary_rate_unit    661
date_posted         836
seniority             0
tiene_skills          0
dtype: int64

## 7. Conversión de tipos  
skill , fecha, salario 

In [23]:
print(df["skills_required"].dtype)
print(df["skills_required"].iloc[0])

str
[".Net", "C#", "React", "TypeScript", "Azure", "CI/CD", "Docker", "Agile"]


Skills: de texto a lista - .replace(), .strip() y limpieza de strings 

In [24]:
# Diccionario de sinónimos: la misma skill escrita de distintas formas
mapa_skills = {
    "reactjs": "react", "react.js": "react", "react js": "react",
    "nodejs": "node.js", "node": "node.js",
    "powerbi": "power bi",
    "cicd": "ci/cd", "ci/cd pipelines": "ci/cd",
    "google cloud": "gcp", "google cloud platform": "gcp",
    "ml": "machine learning",
    "microsoft excel": "excel",
    "aws cloud": "aws", "azure cloud": "azure",
}


def limpiar_skills(texto):
    if pd.isnull(texto):
        return []

    for simbolo in ["[", "]", '"']:
        texto = texto.replace(simbolo, "")

    skills = []
    for skill in texto.split(","):
        skill = skill.strip().lower().replace("‑", "-")
        if skill in mapa_skills:              
            skill = mapa_skills[skill]        
        if skill != "" and skill not in skills:
            skills.append(skill)
    return skills


print(limpiar_skills('["ReactJS", "React", "NodeJS", "PowerBI"]'))

df["skills"] = [limpiar_skills(s) for s in df["skills_required"]]
df["cantidad_skills"] = [len(s) for s in df["skills"]]

df[["title", "rol", "skills", "cantidad_skills"]].head()

['react', 'node.js', 'power bi']


,title,rol,skills,cantidad_skills
685,Full Stack Developer,Full Stack Developer,"[.net, c#, react, typescript, azure, ci/cd, do...",8
884,Senior Full Stack Engineer (Python / React),Full Stack Developer,"[full-stack development, python/flask, react/t...",8
886,Senior Full Stack Engineer (Node / Typescript ...,Full Stack Developer,"[node.js, typescript, react, mysql, aws, docke...",8
887,Full Stack Engineer (Python / React),Full Stack Developer,[],0
1021,Business Intelligence Analyst I,Data Analyst,"[data analysis, bi reporting, trend analysis, ...",4


In [25]:
# ¿En cuántas ofertas aparece cada skill?
for skill in ["python", "java", "javascript", "typescript", "sql"]:
    cantidad = sum([skill in lista for lista in df["skills"]])
    print(f"{skill:12s} → {cantidad} ofertas")

python       → 438 ofertas
java         → 143 ofertas
javascript   → 55 ofertas
typescript   → 88 ofertas
sql          → 301 ofertas


aplicar a todas las ofertas  
04: list comprehension - crear columnas

In [26]:
todas_las_skills = [skill for lista in df["skills"] for skill in lista]

print(f"Total de menciones de skills: {len(todas_las_skills):,}")
print("\nLas 10 skills más mencionadas:")
pd.Series(todas_las_skills).value_counts().head(10)

Total de menciones de skills: 7,728

Las 10 skills más mencionadas:


python                438
sql                   301
ci/cd                 295
kubernetes            202
aws                   181
java                  143
docker                134
react                 127
terraform             127
data visualization     94
Name: count, dtype: int64

### 7.1 Fecha de publicación
clase 09: pariente de pd.to_numeric

In [27]:
print(f"Tipo antes: {df['date_posted'].dtype}")

df["date_posted"] = pd.to_datetime(df["date_posted"], format="%m/%d/%Y", errors="coerce")
print(f"Tipo después: {df['date_posted'].dtype}")

fechas_imposibles = df["date_posted"] > "2026-06-30"
print(f"Fechas imposibles (posteriores a junio 2026): {fechas_imposibles.sum()}")
df.loc[fechas_imposibles, "date_posted"] = pd.NaT

print(f"Rango de fechas: {df['date_posted'].min()} a {df['date_posted'].max()}")

Tipo antes: str
Tipo después: datetime64[us]
Fechas imposibles (posteriores a junio 2026): 1
Rango de fechas: 2023-12-01 00:00:00 a 2026-06-06 00:00:00


## 8. Estado final

In [28]:
print("=== ESTADO FINAL ===")
print(f"Filas: {len(df)}")
print(f"\nOfertas por rol:\n{df['rol'].value_counts()}")
print(f"\nNulos por columna:\n{df.isnull().sum()}")
print(f"\nTipos de datos:\n{df.dtypes}")

=== ESTADO FINAL ===
Filas: 1075

Ofertas por rol:
rol
Cloud Engineer          310
Full Stack Developer    255
Data Analyst            181
QA Engineer             173
Data Scientist          156
Name: count, dtype: int64

Nulos por columna:
title                 0
rol                   0
company_name          0
country               0
city                  0
work_model            0
employment_type       0
skills_required     114
education_level     968
salary_min          666
salary_max          672
salary_currency     599
salary_rate_unit    661
date_posted         857
seniority             0
tiene_skills          0
skills                0
cantidad_skills       0
dtype: int64

Tipos de datos:
title                          str
rol                            str
company_name                   str
country                        str
city                           str
work_model                     str
employment_type                str
skills_required                str
education_level  

### 9 . Salario anual en USD

In [29]:
mapa_periodo = {
    "year": "year", "yearly": "year", "Year": "year", "YEAR": "year", "Yr": "year",
    "Annual": "year", "annual": "year", "Annum": "year", "per annum": "year",
    "month": "month", "monthly": "month", "Monthly": "month",
    "hour": "hour",
}
df["salary_rate_unit"] = df["salary_rate_unit"].map(mapa_periodo)

df["salary_rate_unit"].value_counts(dropna=False)

salary_rate_unit
NaN      661
year     370
hour      22
month     22
Name: count, dtype: int64

In [30]:
multiplicador = {"year": 1, "month": 12, "hour": 2080}

salario_promedio = (df["salary_min"] + df["salary_max"]) / 2
df["salario_anual_usd"] = salario_promedio * df["salary_rate_unit"].map(multiplicador)

# Solo USD, para no mezclar monedas
df.loc[df["salary_currency"] != "USD", "salario_anual_usd"] = None

# Validación: un salario de 0 es imposible (error de carga)
salario_cero = df["salario_anual_usd"] == 0
print(f"Salarios en 0 (imposibles): {salario_cero.sum()}")
df.loc[salario_cero, "salario_anual_usd"] = None

print(f"\nOfertas con salario anual en USD: {df['salario_anual_usd'].notna().sum()}")
df["salario_anual_usd"].describe().round(0)

Salarios en 0 (imposibles): 3

Ofertas con salario anual en USD: 340


count       340.0
mean     142540.0
std       49379.0
min       31200.0
25%      113700.0
50%      131650.0
75%      171735.0
max      313500.0
Name: salario_anual_usd, dtype: float64

In [31]:
columnas_a_sacar = ["skills_required", "salary_min", "salary_max", "salary_currency", "salary_rate_unit"]
salida = df.drop(columns=columnas_a_sacar).copy()
salida["skills"] = ["|".join(lista) for lista in salida["skills"]]

salida.to_csv("../data/processed/ofertas_tech_limpio.csv", index=False)
print(f"Guardado: {len(salida)} filas, {salida.shape[1]} columnas")

Guardado: 1075 filas, 14 columnas


## 10. dataset limpio